### 验证自定义模型有效性
通过相同的数据集进行forward推理，如果lerobot_policy_pi05 与lerobot.policies.pi05 推理结果相同，证明模型架构有效

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"  # 禁止联网，只用本地 cache

In [5]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# 本地数据集：repo_id 为子目录名，root 为父目录
dataset = LeRobotDataset(
    repo_id='/vla/.data/test',
    video_backend='torchcodec',
    )
dataset

LeRobotDataset({
    Repository ID: '/vla/.data/test',
    Number of selected episodes: '1',
    Number of selected samples: '436',
    Features: '['observation.state', 'action', 'observation.images.robot0_agentview_left_image', 'observation.images.robot0_agentview_right_image', 'observation.images.robot0_eye_in_hand_image', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

In [ ]:
import torch
from lerobot.utils.import_utils import register_third_party_plugins
from lerobot.policies.factory import make_policy, make_pre_post_processors
from lerobot_policy_my_policy import MyPolicyConfig
from lerobot_policy_pi05 import PI05Config,PI05Policy
model_id = "/vla/.models/lerobot-pi05_base"
register_third_party_plugins()  # 必须：注册 lerobot_policy_my_policy 插件

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 方式 A：从零初始化（还没有训练好的 checkpoint）
# config = MyPolicyConfig(device=str(device))
config = PI05Config(device=str(device))
policy = PI05Policy.from_pretrained(model_id).to(device).eval() # 2分钟

In [14]:
policy.model.paligemma_with_expert

PaliGemmaWithExpertModel(
  (paligemma): PaliGemmaForConditionalGeneration(
    (model): PaliGemmaModel(
      (vision_tower): SiglipVisionModel(
        (vision_model): SiglipVisionTransformer(
          (embeddings): SiglipVisionEmbeddings(
            (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
            (position_embedding): Embedding(256, 1152)
          )
          (encoder): SiglipEncoder(
            (layers): ModuleList(
              (0-26): 27 x SiglipEncoderLayer(
                (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                (self_attn): SiglipAttention(
                  (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                  (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                  (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                  (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
      

In [ ]:
preprocess, postprocess = make_pre_post_processors(
    policy.config,
    dataset_stats=dataset.meta.stats,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

print("policy type:", policy.config.type)
print("input features:", list(policy.config.input_features.keys()))

### 推理验证

In [ ]:
batch = preprocess(dataset[0])
def to_device(batch):
    return {
        k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }
batch = to_device(batch)

In [ ]:
with torch.inference_mode():
    pred_action_raw = policy.select_action(batch)
pred_action_raw

### 前向传播

In [ ]:
with torch.inference_mode(): # 不进行反向传播
    loss, output_dict = policy.forward(batch)
print("loss:", loss.item() if hasattr(loss, "item") else loss)
print("output keys:", output_dict.keys() if isinstance(output_dict, dict) else type(output_dict))

In [3]:
#!/usr/bin/env python3
import sys
from pathlib import Path
from nbconvert import MarkdownExporter

# p = Path(sys.argv[1])
p = Path("/vla/my_vla/.code/自定义policy有效性证明.ipynb")
p.with_suffix(".md").write_text(
    MarkdownExporter().from_filename(str(p))[0], encoding="utf-8"
)

/mnt/workspace/luyi/.cache/miniconda3/envs/myvla/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


14248

In [2]:
!pip install nbconvert

Looking in indexes: http://mirrors.baidubce.com/pypi/simple/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [nbconvert]17 [nbconvert]]up4]


## lerobot - pi05 - official

In [ ]:
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.factory import make_pre_post_processors

# Swap this import per-policy
from lerobot.policies.pi05 import PI05Policy as PI05PolicyOfficial

# load a policy
model_id = "/vla/.models/lerobot-pi05_base"  # <- swap checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

policy_pi05 = PI05PolicyOfficial.from_pretrained(model_id).to(device).eval()

preprocess_pi05, postprocess_pi05 = make_pre_post_processors(
    policy_pi05.config,
    model_id,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

In [ ]:
policy_pi05

PI05Config(n_obs_steps=1, input_features={'observation.images.robot0_agentview_left_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 'observation.images.robot0_eye_in_hand_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 'observation.images.robot0_agentview_right_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))}, output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))}, device='cuda', use_amp=True, use_peft=False, push_to_hub=False, repo_id=None, private=None, tags=None, license=None, pretrained_path=None, paligemma_variant='gemma_2b', action_expert_variant='gemma_300m', dtype='bfloat16', chunk_size=50, n_action_steps=50, max_state_dim=32, max_action_dim=32, num_inference_steps=10, time_sampling_beta_alpha=1.5, time_sampling_beta_beta=1.0, time_sampling_scale=0.999, time_samp

PI05Config(n_obs_steps=1, 
input_features={'observation.images.robot0_agentview_left_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 'observation.images.robot0_eye_in_hand_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 'observation.images.robot0_agentview_right_image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 224, 224)), 'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))}, 

output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))}, 

device='cuda', use_amp=True, use_peft=False, push_to_hub=False, repo_id=None, private=None, tags=None, license=None, 

pretrained_path=None, paligemma_variant='gemma_2b', action_expert_variant='gemma_300m', dtype='bfloat16', chunk_size=50, n_action_steps=50, max_state_dim=32, max_action_dim=32, num_inference_steps=10, time_sampling_beta_alpha=1.5, time_sampling_beta_beta=1.0, time_sampling_scale=0.999, time_sampling_offset=0.001, min_period=0.004, max_period=4.0, rtc_config=None, image_resolution=(224, 224), empty_cameras=0, tokenizer_max_length=200, normalization_mapping={'VISUAL': <NormalizationMode.IDENTITY: 'IDENTITY'>, 'STATE': <NormalizationMode.QUANTILES: 'QUANTILES'>, 'ACTION': <NormalizationMode.QUANTILES: 'QUANTILES'>}, gradient_checkpointing=False, compile_model=False, compile_mode='max-autotune', 

freeze_vision_encoder=False, train_expert_only=False, 

optimizer_lr=2.5e-05, optimizer_betas=(0.9, 0.95), optimizer_eps=1e-08, optimizer_weight_decay=0.01, optimizer_grad_clip_norm=1.0, scheduler_warmup_steps=1000, scheduler_decay_steps=30000, scheduler_decay_lr=2.5e-06)

In [4]:
policy_pi05.model

NameError: name 'policy_pi05' is not defined